# 🚀 你的第一个 AI Agent：从 Prompt 到 Action

**欢迎来到 Kaggle 5 天 Agents 课程！**

这个 Notebook 是你构建 AI Agent 的第一步。Agent 不仅能响应提示词，还可以**执行动作**来查找信息或完成任务。

在本 Notebook 中，你将：

- ✅ 安装 [Agent Development Kit (ADK)](https://google.github.io/adk-docs/)
- ✅ 配置 API Key 以使用 Gemini 模型
- ✅ 搭建你的第一个简单 Agent
- ✅ 运行 Agent，并观察它如何使用工具（如 Google Search）回答问题

## ⚙️ 第 1 部分：环境准备

### 1.1：安装依赖

Kaggle Notebooks 环境已经预装了 Python 版 [google-adk](https://google.github.io/adk-docs/) 及其所需依赖，因此你在本 Notebook 中无需额外安装包。

如果你要在课程外、自己的 Python 开发环境中安装并使用 ADK，可运行：

```
pip install google-adk
```

### 1.2：配置 Gemini API Key

本 Notebook 使用 [Gemini API](https://ai.google.dev/gemini-api/docs)，需要先完成鉴权。

**1. 获取 API Key**

如果你还没有，可在 [Google AI Studio 创建 API key](https://aistudio.google.com/app/api-keys)。

**2. 将 Key 添加到 Kaggle Secrets**

接下来，你需要把 API Key 作为 Kaggle User Secret 添加到 Notebook 中。

1. 在 Notebook 编辑器顶部菜单选择 `Add-ons` -> `Secrets`。
2. 新建一个标签为 `GOOGLE_API_KEY` 的 secret。
3. 把 API Key 粘贴到 "Value" 字段并点击 "Save"。
4. 确认 `GOOGLE_API_KEY` 左侧复选框已勾选，使其附加到该 Notebook。

**3. 在 Notebook 中鉴权**

运行下面的代码单元完成鉴权。

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


### 1.3：导入 ADK 组件

现在从 Agent Development Kit 和 Generative AI 库中导入接下来会用到的组件。这样可以让代码结构更清晰，并确保我们具备所需的核心构建块。

In [3]:
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.4：辅助函数

我们会定义一些辅助函数。如果你是在 Kaggle 环境之外运行本内容，这一步并非必须。

In [5]:
# Define helper functions that will be reused throughout the notebook

from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers


# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("✅ Helper functions defined.")

✅ Helper functions defined.


### 1.5：配置重试选项

在使用 LLM 时，你可能会遇到速率限制或服务暂时不可用等瞬时错误。重试选项会通过指数退避自动重试请求，从而处理这类失败。

In [9]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

---

## 🤖 第 2 部分：用 ADK 构建你的第一个 AI Agent

### 🤔 2.1 什么是 AI Agent？

你很可能已经用过 Gemini 这类 LLM：给它一个 Prompt，它返回一段文本。

`Prompt -> LLM -> Text`

AI Agent 在此基础上更进一步。Agent 可以思考、执行动作，并观察动作结果，从而给出更好的答案。

`Prompt -> Agent -> Thought -> Action -> Observation -> Final Answer`

在本 Notebook 中，我们将构建一个可以执行 Google 搜索动作的 Agent。来看看差异！

### 2.2 定义你的 Agent

现在开始搭建 Agent。我们将通过设置关键属性来配置 `Agent`，这些属性决定了它做什么以及如何运行。

更多信息可参考 [ADK 中的 agents 文档](https://google.github.io/adk-docs/agents/)。

我们将设置的主要属性有：

- **name** 与 **description**：用于标识 Agent 的名称与描述。
- **model**：驱动 Agent 推理的具体 LLM。这里我们使用 "gemini-2.5-flash-lite"。
- **instruction**：Agent 的指导提示词，定义其目标与行为方式。
- **tools**：Agent 可用的[工具](https://google.github.io/adk-docs/tools/)列表。我们先给它 `google_search` 工具，使其能在线获取最新信息。

In [10]:
root_agent = Agent(
    name="helpful_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant. Use Google Search for current info or if unsure.",
    tools=[google_search],
)

print("✅ Root Agent defined.")

✅ Root Agent defined.


### 2.3 运行你的 Agent

现在让 Agent 真正跑起来，并发送一个查询。你需要一个 [`Runner`](https://google.github.io/adk-docs/runtime/)，它是 ADK 中的核心编排组件：负责管理对话、把消息发送给 Agent，并处理返回结果。

**a. 创建一个 `InMemoryRunner`，并让它使用我们的 `root_agent`：**

In [11]:
runner = InMemoryRunner(agent=root_agent)

print("✅ Runner created.")

✅ Runner created.


👉 注意：我们在这个 Notebook 中直接使用 Python Runner。你也可以使用 ADK 命令行工具运行 Agent，例如 `adk run`、`adk web` 或 `adk api_server`。更多内容请参阅 [ADK runtime 文档](https://google.github.io/adk-docs/runtime/)。

**b. 现在你可以调用 `.run_debug()` 方法发送 Prompt 并获取答案。**

👉 该方法对 session 的创建与维护做了封装，常用于原型开发。我们会在第 3 天详细讲解“什么是 session 以及如何创建 session”。

In [12]:
response = await runner.run_debug(
    "What is Agent Development Kit from Google? What languages is the SDK available in?"
)


 ### Created new session: debug_session_id

User > What is Agent Development Kit from Google? What languages is the SDK available in?
helpful_assistant > The Google Agent Development Kit (ADK) is a flexible, modular, and open-source framework designed to simplify the development and deployment of AI agents. It aims to make agent development feel more akin to traditional software development, allowing developers to create, deploy, and orchestrate agentic architectures with greater ease. The ADK is model-agnostic and deployment-agnostic, meaning it can be used with various AI models and deployed in different environments. It's optimized for Gemini and the Google ecosystem but is compatible with other frameworks as well.

The SDK is available in the following programming languages:
*   Python
*   Java
*   Go
*   TypeScript

Support for more languages is planned for the future.


你可以在响应中看到 ADK 的概述及其支持的语言信息。

### 2.4 它是如何工作的？

Agent 通过 Google Search 获取了 ADK 的最新信息，它之所以知道要用这个工具，是因为：

1. Agent 会检查并了解自己有哪些可用工具。
2. Agent 的 instruction 明确要求：在需要最新信息或不确定答案时使用搜索工具。

要查看 Agent 思考与动作的完整细粒度轨迹，最佳方式是使用 **ADK Web UI**，我们会在本 Notebook 后面进行配置。

课程后续也会介绍更完整的日志与可观测性工作流。

### 🚀 2.5 轮到你了！

现在你可以亲自体验 Agent 的能力。请问它一个需要“当前信息”的问题。

你可以试试下面这些，或自己发挥：

- 伦敦现在天气怎么样？
- 上一届足球世界杯冠军是谁？
- 现在影院正在上映哪些新电影？

In [13]:
response = await runner.run_debug("What's the weather in London?")


 ### Continue session: debug_session_id

User > What's the weather in London?
helpful_assistant > The weather in London is currently mostly cloudy, with a temperature of 35°F (2°C). It feels like 27°F (-3°C) and the humidity is around 81%.

Looking ahead, tonight is expected to have rain and snow, with a 20% chance of snow. Tomorrow, Friday, will be sunny during the day and clear at night, with a 15% chance of snow. The temperature on Friday will range between 30°F (-1°C) and 39°F (4°C).

The 10-day forecast for London indicates a mix of weather conditions, including sunny days, partly cloudy skies, and chances of rain and snow. Temperatures are generally expected to be cool, with daily highs ranging from the low 30s to mid-40s Fahrenheit (0-8°C) and overnight lows often dropping below freezing.


---

## 💻 第 3 部分：体验 ADK Web 界面

### 概览

ADK 内置了一个 Web 界面，可用于与 Agent 交互聊天、测试与调试。

<img width="1200" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/adk-web-ui.gif" alt="ADK Web UI" />

要使用 ADK Web UI，你需要先通过 `adk create` 命令生成一个基于 Python 文件的 Agent。

运行下面命令会创建一个 `sample-agent` 文件夹，其中包含所需全部文件：包括用于编写代码的 `agent.py`、预配置 API Key 的 `.env` 文件，以及 `__init__.py` 文件：

In [14]:
!adk create sample-agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY


Agent created in /kaggle/working/sample-agent:
- .env
- __init__.py
- agent.py



在 Kaggle Notebooks 环境中获取用于访问 ADK Web UI 的专属 URL：

In [15]:
url_prefix = get_adk_proxy_url()

现在我们可以运行 ADK Web：

In [ ]:
!adk web --url_prefix {url_prefix}

/usr/local/lib/python3.11/dist-packages/google/adk/cli/fast_api.py:130: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.11/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
INFO:     Started server process [107]
INFO:     Waiting for application startup.

+-----------------------------------------------------------------------------+
| ADK Web Server started                                                      |
|                                                                             |
| For local testing, access at http:

现在你可以通过上方链接访问 ADK 开发界面。

打开链接后，你会看到 ADK Web 界面，并可以在其中向你的 ADK Agent 提问。

注意：这个示例 Agent 没有启用任何工具（例如 Google Search）。它是一个基础 Agent，主要用于帮助你探索 UI 功能。

‼️ **重要：不要把代理链接分享给任何人**。该链接 URL 中包含你的鉴权令牌，属于敏感信息。

---

## ✅ 恭喜！

你已经用 ADK 构建并运行了第一个 Agent！你刚刚亲眼看到 Agent 开发的核心概念如何落地。

最关键的收获是：你的 Agent 不只是*回答*问题，它会先**推理**自己需要更多信息，然后通过工具去**行动**。这种“可行动”能力正是 Agent 型 AI 的基础。

**ℹ️ 说明：无需提交！**

这个 Notebook 仅用于动手实践与学习。完成课程**不需要**把它提交到任何地方。

### 📚 继续学习

你可以参考以下文档深入了解：

- [ADK Documentation](https://google.github.io/adk-docs/)
- [ADK Quickstart for Python](https://google.github.io/adk-docs/get-started/python/)
- [ADK Agents Overview](https://google.github.io/adk-docs/agents/)
- [ADK Tools Overview](https://google.github.io/adk-docs/tools/)

### 🎯 下一步

准备好迎接下一个挑战了吗？继续学习下一个 Notebook，了解如何**设计多 Agent 系统**。